# 03 · Video preprocessing status and data checks

**Adapted:** The videos have already been converted to MediaPipe arrays. This notebook validates the current arrays and summarizes the frozen split; it does not recreate RGB frames or rerun video processing.


In [ ]:
from pathlib import Path
import pandas as pd
ROOT=Path.cwd().resolve()
while not (ROOT/'data/processed/WLASL100/all_metadata.csv').exists() and ROOT.parent!=ROOT: ROOT=ROOT.parent
DATA=ROOT/'data/processed/WLASL100'
if not (DATA/'all_metadata.csv').exists(): raise FileNotFoundError('Open notebook from the sign-language-recognition repository')
import numpy as np
split=pd.read_csv(DATA/'split_v2.csv',dtype={'video_id':str})
rows=[]
for src in ('landmarks_repo','landmarks_kaggle'):
 idx=pd.read_csv(DATA/src/'index.csv',dtype={'video_id':str})
 for r in idx.itertuples():
  p=DATA/src/f'{str(r.video_id).zfill(5)}.npy'
  a=np.load(p,mmap_mode='r')
  rows.append((str(r.video_id).zfill(5),src,a.shape[0],a.shape[1],a.dtype.name))
q=pd.DataFrame(rows,columns=['video_id','source','frames','features','dtype']).merge(split,on='video_id',validate='one_to_one')
assert (q.features==225).all() and (q.dtype=='float32').all()
print(q.groupby(['split','source']).agg(videos=('video_id','count'),median_frames=('frames','median')))
